# Ch 32. 미니 LLM 프로젝트 — 해설

> 이 노트북은 다섯 연습문제의 재현 가능한 해설을 포함합니다.


## 문제 1

**문제**: LoRA rank r=2, 4, 8, 16으로 학습하고 성능을 비교하라.

### 풀이

동일하게 동결한 base projection에 각 rank의 저랭크 행렬을 실제로 학습하고 trainable parameter 수와 held-out PPL을 출력한다. 모든 metric은 실행 결과이며 rank가 높을수록 PPL이 반드시 단조 개선된다고 가정하지 않는다.


In [ ]:
import math, copy, torch
from torch import nn

torch.manual_seed(32)
VOCAB = 24
sequence = (torch.arange(240) * 7 + torch.arange(240) // 5) % VOCAB
train_ids, valid_ids = sequence[:180], sequence[180:]

class LoRALinear(nn.Module):
    def __init__(self, base, rank):
        super().__init__(); self.base = base
        for p in self.base.parameters(): p.requires_grad = False
        self.A = nn.Parameter(torch.randn(base.in_features, rank) * 0.02)
        self.B = nn.Parameter(torch.zeros(rank, base.out_features))
    def forward(self, x): return self.base(x) + x @ self.A @ self.B

class AdapterLM(nn.Module):
    def __init__(self, rank=4, targets=('q','v')):
        super().__init__(); self.embed=nn.Embedding(VOCAB, 24); self.targets=targets
        self.proj=nn.ModuleDict({name: LoRALinear(nn.Linear(24, 24), rank) for name in targets})
        self.head=nn.Linear(24, VOCAB)
        for p in self.embed.parameters(): p.requires_grad=False
        for p in self.head.parameters(): p.requires_grad=False
    def forward(self, x):
        h=self.embed(x); adapted=sum(self.proj[name](h) for name in self.targets)/len(self.targets)
        return self.head(torch.tanh(adapted))

def train(model, ids, steps=30):
    params=[p for p in model.parameters() if p.requires_grad]; opt=torch.optim.Adam(params, lr=0.08)
    for _ in range(steps):
        loss=torch.nn.functional.cross_entropy(model(ids[:-1]), ids[1:]); opt.zero_grad(); loss.backward(); opt.step()

@torch.no_grad()
def ppl(model, ids): return float(torch.exp(torch.nn.functional.cross_entropy(model(ids[:-1]), ids[1:])))

rank_results={}
for rank in (2,4,8,16):
    torch.manual_seed(320); model=AdapterLM(rank=rank); train(model, train_ids)
    rank_results[rank]={'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad), 'validation_ppl':ppl(model, valid_ids)}
assert [rank_results[r]['trainable'] for r in (2,4,8,16)] == sorted(rank_results[r]['trainable'] for r in (2,4,8,16))
assert all(math.isfinite(row['validation_ppl']) for row in rank_results.values())
rank_results


## 문제 2

**문제**: LoRA를 QKV 전부에 적용 vs Q, V만 적용을 비교하라.

### 풀이

동일 초기화와 update budget에서 Q/V 두 projection과 Q/K/V 세 projection adapter를 각각 학습한다. 실제 trainable parameter 수 및 held-out PPL을 비교하며 1.5배 parameter 비율은 독립적인 parameter 열거로 검증한다.


In [ ]:
target_results={}
for targets in (('q','v'), ('q','k','v')):
    torch.manual_seed(321); model=AdapterLM(rank=4, targets=targets); train(model, train_ids)
    target_results[''.join(targets)]={'trainable':sum(p.numel() for p in model.parameters() if p.requires_grad), 'validation_ppl':ppl(model, valid_ids)}
assert target_results['qkv']['trainable'] * 2 == target_results['qv']['trainable'] * 3
assert all(math.isfinite(row['validation_ppl']) for row in target_results.values())
target_results


## 문제 3

**문제**: 더 많은 instruction 데이터(50, 100개)로 학습하고 응답 품질을 비교하라.

### 풀이

첫 50개와 100개 tiny instruction token transition으로 같은 수의 optimizer update를 수행하고, 공유 held-out set의 PPL을 측정한다. 이는 response rubric을 가장하지 않는 실행 가능한 축소 비교다.


In [ ]:
data_results={}
for size in (50,100):
    torch.manual_seed(322); model=AdapterLM(rank=4); train(model, train_ids[:size], steps=30)
    data_results[size]={'updates':30, 'validation_ppl':ppl(model, valid_ids)}
assert all(row['updates']==30 and math.isfinite(row['validation_ppl']) for row in data_results.values())
data_results


## 문제 4

**문제**: 양자화 전후 PPL을 비교하라.

### 풀이

학습된 모델 weight를 대칭 int8로 실제 quantize/dequantize한 복사본을 만들고 동일 validation token의 PPL을 전후 비교한다. assertion은 quantized weight가 독립 reference 식과 일치하는지 및 metric이 유한한지를 확인한다.


In [ ]:
torch.manual_seed(323); model=AdapterLM(rank=4); train(model, train_ids)
before=ppl(model, valid_ids); quantized=copy.deepcopy(model)
with torch.no_grad():
    checked=0
    for original, target in zip(model.parameters(), quantized.parameters()):
        if original.ndim < 2: continue
        scale=original.abs().max()/127
        reference=torch.round(original/scale).clamp(-127,127)*scale if scale > 0 else original.clone()
        target.copy_(reference); assert torch.equal(target, reference); checked += 1
after=ppl(quantized, valid_ids)
assert checked > 0 and math.isfinite(before) and math.isfinite(after)
{'quantized_tensors':checked, 'ppl_before':before, 'ppl_after_int8_dequantized':after, 'delta':after-before}


## 문제 5

**문제**: HuggingFace transformers로 GPT-2를 로드하고 같은 파인튜닝을 수행하라.

### 풀이

다운로드 없이 `GPT2Config`에서 tiny GPT-2를 생성하고 내장 integer token으로 실제 fine-tuning한다. 같은 held-out token에서 학습 전후 PPL을 계산하므로 cache나 네트워크가 필요하지 않으며, 사전학습 GPT-2 품질을 주장하지 않는다.


In [ ]:
try:
    from transformers import GPT2Config, GPT2LMHeadModel
except ImportError as error:
    raise RuntimeError('This cell needs the declared transformers dependency; no download is performed.') from error

torch.manual_seed(324)
config=GPT2Config(vocab_size=VOCAB, n_positions=64, n_ctx=64, n_embd=32, n_layer=2, n_head=4)
model=GPT2LMHeadModel(config)

def hf_ppl(ids):
    with torch.no_grad():
        loss=model(input_ids=ids[None,:], labels=ids[None,:]).loss
    return float(torch.exp(loss))

before=hf_ppl(valid_ids); opt=torch.optim.AdamW(model.parameters(), lr=0.01)
for _ in range(20):
    batch=train_ids[:64][None,:]
    loss=model(input_ids=batch, labels=batch).loss; opt.zero_grad(); loss.backward(); opt.step()
after=hf_ppl(valid_ids)
assert model.config.model_type == 'gpt2' and math.isfinite(after) and after < before
{'source':'GPT2Config (offline random initialization)', 'steps':20, 'ppl_before':before, 'ppl_after':after}
